## Pull all data and store into GCP using Ticker

In [47]:
# Get CIK from TICKER
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json

BASE = "https://data.sec.gov"
HEADERS = {"User-Agent": "kurio-agent/1.0 (potatojacket9@gmail.com)"}

## Step 1. Get CIK for the input ticker symbol
def get_cik(ticker):
    """Retrieve CIK for a given ticker symbol via the SEC company tickers map."""
    ticker = (ticker or "").strip().upper()
    if not ticker:
        return None

    lookup = requests.get(
        "https://www.sec.gov/files/company_tickers.json",
        headers=HEADERS,
    )
    lookup.raise_for_status()
    data = lookup.json()

    for _, company in data.items():
        if str(company.get("ticker", "")).upper() == ticker:
            return str(company["cik_str"]).zfill(10)
    return None

## Step 2. Get 10-K filing URL for the CIK (latest, or a specific report year)
def get_latest_10k_url(cik, year=None):
    """
    Retrieve a 10-K filing document URL and dates for a given CIK.
    If year is provided, prefer the filing whose reportDate starts with that year.
    Otherwise return the most recent 10-K.
    """
    cik = str(cik).zfill(10)
    url = f"{BASE}/submissions/CIK{cik}.json"
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    data = res.json()

    recent = data["filings"]["recent"]
    target_year = str(year) if year is not None else None

    for form, acc, filing_date, report_date in zip(
        recent["form"],
        recent["accessionNumber"],
        recent["filingDate"],
        recent["reportDate"],
    ):
        if form != "10-K":
            continue
        if target_year and not str(report_date).startswith(target_year):
            continue

        acc_num = acc.replace("-", "")
        filing_url = (
            f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_num}/{acc}-index.html"
        )
        return filing_url, filing_date, report_date

    return None, None, None


### [A] Get financial statements - income statement, cashflow statement - from 10K

#### [A1] Supporting functions for income statement, cashflow statement

In [49]:
## Step 3: Find actual XBRL XML file from 10K URL
from datetime import date, datetime
from collections import defaultdict  # kept for notebook reuse / debugging


def _local_name(name):
    """Normalize BeautifulSoup tag names across prefix / Clark notation."""
    if not name:
        return ""
    if name.startswith("{") and "}" in name:
        return name.rsplit("}", 1)[-1]
    if ":" in name:
        return name.split(":", 1)[-1]
    return name


def _find_all_by_local_name(root, local_names):
    wanted = {n.lower() for n in local_names}
    matches = []
    for el in root.find_all(True):
        if _local_name(el.name).lower() in wanted:
            matches.append(el)
    return matches


def _parse_date(text):
    if not text:
        return None
    if isinstance(text, date) and not isinstance(text, datetime):
        return text
    if isinstance(text, datetime):
        return text.date()
    text = str(text).strip()[:10]
    try:
        return datetime.strptime(text, "%Y-%m-%d").date()
    except ValueError:
        return None


def _parse_contexts(soup):
    """
    Build context_id -> metadata for period selection.
    Prefers consolidated facts (no dimensional segment members).
    """
    contexts = {}
    for ctx in _find_all_by_local_name(soup, ["context"]):
        ctx_id = ctx.get("id")
        if not ctx_id:
            continue

        start_el = next(iter(_find_all_by_local_name(ctx, ["startDate"])), None)
        end_el = next(iter(_find_all_by_local_name(ctx, ["endDate"])), None)
        instant_el = next(iter(_find_all_by_local_name(ctx, ["instant"])), None)
        segment_el = next(iter(_find_all_by_local_name(ctx, ["segment"])), None)
        members = _find_all_by_local_name(ctx, ["explicitMember", "typedMember"])

        start = _parse_date(start_el.get_text() if start_el else None)
        end = _parse_date(end_el.get_text() if end_el else None)
        instant = _parse_date(instant_el.get_text() if instant_el else None)
        has_dimensions = segment_el is not None or len(members) > 0

        duration_days = (end - start).days if start and end else None

        contexts[ctx_id] = {
            "id": ctx_id,
            "start": start,
            "end": end,
            "instant": instant,
            "duration_days": duration_days,
            "has_dimensions": has_dimensions,
            "is_duration": start is not None and end is not None,
            "is_instant": instant is not None,
        }
    return contexts


def _pick_annual_duration_context_ids(contexts, report_date):
    """
    Prefer consolidated (~1 year) duration contexts ending on the 10-K report date.
    Returns an ordered list of acceptable context ids (best first).
    """
    report = _parse_date(report_date)
    candidates = []

    for ctx in contexts.values():
        if not ctx["is_duration"] or ctx["has_dimensions"]:
            continue
        days = ctx["duration_days"]
        if days is None or days < 300 or days > 400:
            continue

        end_match = 0
        if report and ctx["end"]:
            delta = abs((ctx["end"] - report).days)
            if delta == 0:
                end_match = 3
            elif delta <= 7:
                end_match = 2
            elif ctx["end"].year == report.year:
                end_match = 1

        year_fit = -abs((days or 365) - 365)
        candidates.append((end_match, year_fit, days, ctx["id"]))

    candidates.sort(reverse=True)
    if candidates:
        return [c[3] for c in candidates]

    # Fallback: any consolidated duration ending nearest to report_date
    fallback = []
    for ctx in contexts.values():
        if not ctx["is_duration"] or ctx["has_dimensions"]:
            continue
        if report and ctx["end"]:
            score = -abs((ctx["end"] - report).days)
        else:
            score = ctx["duration_days"] or 0
        fallback.append((score, ctx["id"]))
    fallback.sort(reverse=True)
    return [c[1] for c in fallback]


def _pick_instant_context_ids(contexts, target_date, window_days=7):
    """Prefer consolidated instant contexts on/near a balance-sheet date."""
    target = _parse_date(target_date)
    if not target:
        return []

    candidates = []
    for ctx in contexts.values():
        if not ctx["is_instant"] or ctx["has_dimensions"]:
            continue
        delta = abs((ctx["instant"] - target).days)
        if delta <= window_days:
            candidates.append((-delta, ctx["id"]))
    candidates.sort(reverse=True)
    return [c[1] for c in candidates]


def _numeric_from_element(el):
    """Parse XBRL / iXBRL numeric facts, honoring scale when present."""
    if el is None:
        return None
    raw = (el.get_text() or "").strip().replace(",", "").replace(" ", "")
    if not raw or raw in {"—", "-", "–", "−"}:
        return None

    sign = -1.0 if (el.get("sign") or "").strip() == "-" else 1.0

    try:
        value = float(raw)
    except ValueError:
        return None

    scale_attr = el.get("scale")
    if scale_attr is not None and str(scale_attr).strip() != "":
        try:
            value *= 10 ** int(scale_attr)
        except ValueError:
            pass

    return sign * value


def _extract_best_fact_value(soup, tag_list, preferred_context_ids, prefer_max=False):
    """
    Extract a numeric fact for tags that appear in a preferred context.
    If prefer_max=True (Total Revenue), pick largest abs value among preferred contexts.
    Does not fall back to unscoped first-hit (avoids segment/prior-year bugs).
    """
    if not preferred_context_ids:
        return None

    preferred_rank = {cid: idx for idx, cid in enumerate(preferred_context_ids)}
    local_by_tag = [tag.split(":")[-1].lower() for tag in tag_list]
    allowed_locals = set(local_by_tag)

    # Pre-index facts by local name for speed
    facts_by_local = defaultdict(list)
    for el in soup.find_all(True):
        ln = _local_name(el.name).lower()
        if ln in allowed_locals and (el.get("contextRef") or el.get("contextref")):
            facts_by_local[ln].append(el)

    best = None  # (rank, abs_value_or_0, value, tag_index)

    for tag_index, local in enumerate(local_by_tag):
        for el in facts_by_local.get(local, []):
            ctx_id = el.get("contextRef") or el.get("contextref")
            if ctx_id not in preferred_rank:
                continue

            value = _numeric_from_element(el)
            if value is None:
                continue

            rank = preferred_rank[ctx_id]
            candidate = (rank, abs(value) if prefer_max else 0, value, tag_index)

            if best is None:
                best = candidate
                continue

            if prefer_max:
                if candidate[0] < best[0] or (
                    candidate[0] == best[0] and candidate[1] > best[1]
                ):
                    best = candidate
            else:
                if candidate[0] < best[0] or (
                    candidate[0] == best[0] and candidate[3] < best[3]
                ):
                    best = candidate

    return best[2] if best is not None else None


def get_primary_xbrl_url(index_url):
    """Find the main XBRL instance XML inside the 10-K index page."""
    res = requests.get(index_url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    candidates = []
    for link in soup.find_all("a", href=True):
        href = link["href"]
        href_l = href.lower()
        if not href_l.endswith(".xml"):
            continue
        if any(x in href_l for x in ["_cal.xml", "_lab.xml", "_pre.xml", "_def.xml"]):
            continue
        if "xsl" in href_l:
            continue

        filename = href_l.rsplit("/", 1)[-1]
        score = 0
        if any(ch.isdigit() for ch in filename):
            score += 2
        if "-" in filename:
            score += 1
        candidates.append((score, href))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    href = candidates[0][1]
    if href.startswith("http"):
        return href
    return "https://www.sec.gov" + href


def parse_cashflow_from_xbrl(soup, report_date=None):
    """Extract consolidated annual cash-flow values for the 10-K report period."""
    contexts = _parse_contexts(soup)
    duration_ids = _pick_annual_duration_context_ids(contexts, report_date)

    report = _parse_date(report_date)
    period_start = None
    if duration_ids and contexts.get(duration_ids[0], {}).get("start"):
        period_start = contexts[duration_ids[0]]["start"]
    elif report:
        try:
            period_start = date(report.year - 1, report.month, report.day)
        except ValueError:
            period_start = date(report.year - 1, report.month, 28)

    end_instant_ids = _pick_instant_context_ids(contexts, report)
    start_instant_ids = _pick_instant_context_ids(contexts, period_start)

    tags = {
        "Net profit (or loss if negative)": ["us-gaap:NetIncomeLoss"],
        "Depreciation (wear & tear on assets)": [
            "us-gaap:DepreciationDepletionAndAmortization",
            "us-gaap:DepreciationAndAmortization",
        ],
        "stock_comp": ["us-gaap:ShareBasedCompensation"],
        "change_ar": ["us-gaap:IncreaseDecreaseInAccountsReceivable"],
        "change_inventory": [
            "us-gaap:IncreaseDecreaseInInventories",
            "us-gaap:IncreaseDecreaseInInventory",
        ],
        "change_ap": ["us-gaap:IncreaseDecreaseInAccountsPayable"],
        "Cash from day-to-day business (Operating Cashflow)": [
            "us-gaap:NetCashProvidedByUsedInOperatingActivities"
        ],
        "Buying equipment/buildings (Capital Expenditure)": [
            "us-gaap:PaymentsToAcquirePropertyPlantAndEquipment"
        ],
        "acquisitions": ["us-gaap:PaymentsToAcquireBusinessesNetOfCashAcquired"],
        "asset_sales": ["us-gaap:ProceedsFromSaleOfPropertyPlantAndEquipment"],
        "investments_purchase": [
            "us-gaap:PaymentsToAcquireMarketableSecurities",
            "us-gaap:PaymentsToAcquireAvailableForSaleSecuritiesDebt",
        ],
        "investments_maturity": [
            "us-gaap:ProceedsFromMaturitiesOfMarketableSecurities",
            "us-gaap:ProceedsFromSaleAndMaturityOfMarketableSecurities",
        ],
        "Cash from investments (Buying/Selling assets)": [
            "us-gaap:NetCashProvidedByUsedInInvestingActivities"
        ],
        "Money raised from issuing new shares": ["us-gaap:ProceedsFromIssuanceOfCommonStock"],
        "Money spent buying back shares of company": [
            "us-gaap:PaymentsForRepurchaseOfCommonStock"
        ],
        "Borrowed money (New loans or bonds)": [
            "us-gaap:ProceedsFromIssuanceOfLongTermDebt",
            "us-gaap:ProceedsFromDebtNetOfIssuanceCosts",
        ],
        "Loan repayments": ["us-gaap:RepaymentsOfLongTermDebt"],
        "Dividends paid to shareholders": [
            "us-gaap:PaymentsOfDividends",
            "us-gaap:PaymentsOfDividendsCommonStock",
        ],
        "Cash from investors and loans (Financing activities)": [
            "us-gaap:NetCashProvidedByUsedInFinancingActivities"
        ],
        "Change in cash during the period": [
            "us-gaap:CashAndCashEquivalentsPeriodIncreaseDecrease",
            "us-gaap:CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalentsPeriodIncreaseDecreaseIncludingExchangeRateEffect",
        ],
        "Cash at the beginning of the period": [
            "us-gaap:CashAndCashEquivalentsAtCarryingValue",
            "us-gaap:CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
        ],
        "Cash remaining at the end of the period": [
            "us-gaap:CashAndCashEquivalentsAtCarryingValue",
            "us-gaap:CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents",
        ],
    }

    instant_keys = {
        "Cash at the beginning of the period": start_instant_ids,
        "Cash remaining at the end of the period": end_instant_ids,
    }

    data = {}
    for key, tag_list in tags.items():
        preferred = instant_keys.get(key, duration_ids)
        data[key] = _extract_best_fact_value(
            soup,
            tag_list,
            preferred,
            prefer_max=False,
        )
    return data


def parse_income_from_xbrl(soup, report_date=None):
    """
    Extract consolidated annual income-statement values for the 10-K report period.
    Uses contextRef / period filtering so segment and prior-year facts are skipped.
    """
    contexts = _parse_contexts(soup)
    duration_ids = _pick_annual_duration_context_ids(contexts, report_date)

    tags = {
        "revenue": {
            "Total Revenue": [
                "us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax",
                "us-gaap:RevenueFromContractWithCustomerIncludingAssessedTax",
                "us-gaap:Revenues",
                "us-gaap:SalesRevenueNet",
                "us-gaap:SalesRevenueGoodsNet",
                "us-gaap:SalesRevenueServicesNet",
            ],
            "Advertising Revenue": ["us-gaap:AdvertisingRevenue"],
            "Interest Income": [
                "us-gaap:InterestIncomeOperating",
                "us-gaap:InvestmentIncomeInterest",
                "us-gaap:InterestAndDividendIncomeOperating",
                "us-gaap:InterestIncomeOther",
                "us-gaap:InterestIncome",
            ],
            "Other Income": [
                "us-gaap:OtherNonoperatingIncomeExpense",
                "us-gaap:NonoperatingIncomeExpense",
            ],
        },
        "expenses": {
            "Cost of Revenue": [
                "us-gaap:CostOfRevenue",
                "us-gaap:CostOfGoodsAndServicesSold",
                "us-gaap:CostOfGoodsSold",
            ],
            "Research & Development": ["us-gaap:ResearchAndDevelopmentExpense"],
            "Sales & Marketing": [
                "us-gaap:SellingAndMarketingExpense",
                "us-gaap:SellingAndMarketingExpenses",
            ],
            "General & Administrative": [
                "us-gaap:GeneralAndAdministrativeExpense",
                "us-gaap:SellingGeneralAndAdministrativeExpense",
            ],
            "Operating Expenses (Total)": ["us-gaap:OperatingExpenses"],
            "Interest Expense": [
                "us-gaap:InterestExpense",
                "us-gaap:InterestExpenseNonoperating",
            ],
            "Income Tax Expense": ["us-gaap:IncomeTaxExpenseBenefit"],
        },
        "profit": {
            "Gross Profit": ["us-gaap:GrossProfit"],
            "Operating Income": ["us-gaap:OperatingIncomeLoss"],
            "Income Before Tax": [
                "us-gaap:IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
                "us-gaap:IncomeLossFromContinuingOperationsBeforeIncomeTaxesMinorityInterestAndIncomeLossFromEquityMethodInvestments",
                "us-gaap:IncomeLossFromContinuingOperationsBeforeIncomeTaxesDomestic",
            ],
            "Net Income": [
                "us-gaap:NetIncomeLoss",
                "us-gaap:ProfitLoss",
                "us-gaap:NetIncomeLossAvailableToCommonStockholdersBasic",
            ],
        },
        "shares": {
            "Earnings per Share (Basic)": ["us-gaap:EarningsPerShareBasic"],
            "Earnings per Share (Diluted)": ["us-gaap:EarningsPerShareDiluted"],
            "Weighted Average Shares Outstanding (Basic)": [
                "us-gaap:WeightedAverageNumberOfSharesOutstandingBasic"
            ],
            "Weighted Average Shares Outstanding (Diluted)": [
                "us-gaap:WeightedAverageNumberOfDilutedSharesOutstanding"
            ],
        },
    }

    grouped_data = {}
    for section, section_tags in tags.items():
        grouped_data[section] = {}
        for label, tag_list in section_tags.items():
            grouped_data[section][label] = _extract_best_fact_value(
                soup,
                tag_list,
                duration_ids,
                prefer_max=(label == "Total Revenue"),
            )

    rev = grouped_data["revenue"].get("Total Revenue")
    cost = grouped_data["expenses"].get("Cost of Revenue")
    gross = grouped_data["profit"].get("Gross Profit")
    op_exp = grouped_data["expenses"].get("Operating Expenses (Total)")
    r_and_d = grouped_data["expenses"].get("Research & Development")
    s_and_m = grouped_data["expenses"].get("Sales & Marketing")
    g_and_a = grouped_data["expenses"].get("General & Administrative")
    op_inc = grouped_data["profit"].get("Operating Income")

    if gross is None and rev is not None and cost is not None:
        grouped_data["profit"]["Gross Profit"] = rev - cost

    if op_exp is None and any(v is not None for v in [r_and_d, s_and_m, g_and_a]):
        grouped_data["expenses"]["Operating Expenses (Total)"] = sum(
            v for v in [r_and_d, s_and_m, g_and_a] if v is not None
        )

    gross = grouped_data["profit"].get("Gross Profit")
    op_exp = grouped_data["expenses"].get("Operating Expenses (Total)")
    if op_inc is None and gross is not None and op_exp is not None:
        grouped_data["profit"]["Operating Income"] = gross - op_exp
        op_inc = grouped_data["profit"]["Operating Income"]

    inc_before_tax = grouped_data["profit"].get("Income Before Tax")
    int_income = grouped_data["revenue"].get("Interest Income")
    int_exp = grouped_data["expenses"].get("Interest Expense")
    other_inc = grouped_data["revenue"].get("Other Income")
    if inc_before_tax is None and op_inc is not None:
        total_other = (int_income or 0) - (int_exp or 0) + (other_inc or 0)
        grouped_data["profit"]["Income Before Tax"] = op_inc + total_other

    return grouped_data


def parse_cashflow_from_10k(latest_10k_filing_XBRL_XML, report_date=None):
    """Extract cash flow data from a 10-K XBRL or XML filing."""
    response = requests.get(latest_10k_filing_XBRL_XML, headers=HEADERS)
    parser = "xml" if "<?xml" in response.text[:200].lower() else "html.parser"
    soup = BeautifulSoup(response.text, features=parser)
    return parse_cashflow_from_xbrl(soup, report_date=report_date)


def parse_income_from_10k(latest_10k_filing_XBRL_XML, report_date=None):
    """Extract income statement data from a 10-K XBRL or XML filing."""
    response = requests.get(latest_10k_filing_XBRL_XML, headers=HEADERS)
    parser = "xml" if "<?xml" in response.text[:200].lower() else "html.parser"
    soup = BeautifulSoup(response.text, features=parser)
    return parse_income_from_xbrl(soup, report_date=report_date)


#### [A2] Full flow of all supporting functions to return cashflow and income statements

In [51]:
### Consolidate all of the above into a flow:
def get_financials_json(ticker):
    """End-to-end: from ticker -> CIK -> 10-K -> XBRL -> income + cashflow data."""
    ## Step 1. Get CIK for the input ticker symbol
    cik = get_cik(ticker)
    if not cik:
        raise ValueError("CIK not found.")

    ## Step 2. Get latest 10K url for the CIK (ticker)
    filing_url, filing_date, report_date = get_latest_10k_url(cik)
    if not filing_url:
        raise ValueError("No 10-K filing found.")

    print(f"Found 10-K filing: {filing_url} (filed {filing_date}, for {report_date})")

    # Step 3: Find actual XBRL XML file from 10K URL
    xbrl_url = get_primary_xbrl_url(filing_url)
    if not xbrl_url:
        raise ValueError("No XBRL XML file found in filing.")

    print(f"Using XBRL XML file: {xbrl_url}")

    # Step 5a/5b: Parse using report_date so we select the consolidated annual period
    cashflow_data = parse_cashflow_from_10k(xbrl_url, report_date=report_date)
    if not cashflow_data:
        raise ValueError("Cash flow statement not found in filing.")

    income_data = parse_income_from_10k(xbrl_url, report_date=report_date)
    if not income_data:
        raise ValueError("income statement not found in filing.")

    return {
        "ticker": ticker,
        "cik": cik,
        "source": xbrl_url,
        "cashflow": cashflow_data,
        "income_statement": income_data,
        "filing_date": filing_date,
        "report_date": report_date,
    }


### [A3] Store statements into firebase storage and the pointer to this in firestore

In [53]:
import json
import os
from dotenv import load_dotenv
from google.cloud import firestore, storage

# Load env vars (local dev only; safe to keep for API)
load_dotenv()

SERVICE_ACCOUNT_PATH = os.getenv("FIREBASE_SERVICE_ACCOUNT_JSON")
BUCKET_NAME = os.getenv("GCP_STORAGE_BUCKET")

if not BUCKET_NAME:
    raise RuntimeError("Missing GCP_STORAGE_BUCKET in environment")

# --- Client initialization ---
# Local dev: use service account JSON
# Production (Cloud Run / Functions): use default credentials
if SERVICE_ACCOUNT_PATH:
    db = firestore.Client.from_service_account_json(SERVICE_ACCOUNT_PATH)
    storage_client = storage.Client.from_service_account_json(SERVICE_ACCOUNT_PATH)
else:
    db = firestore.Client()
    storage_client = storage.Client()


def upload_json_to_firebase_storage(
    ticker: str,
    year: int,
    statement_type: str,   # "incomeStatement" | "cashFlow"
    data: dict,
):
    """
    Uploads a financial statement dict as JSON to Firebase Storage.

    Storage path:
      gs://<bucket>/filings/{ticker}/{year}/statements/{statement_type}.json
    """
    bucket = storage_client.bucket(BUCKET_NAME)

    object_path = f"filings/{ticker}/{year}/statements/{statement_type}.json"
    blob = bucket.blob(object_path)

    blob.upload_from_string(
        json.dumps(data, ensure_ascii=False),
        content_type="application/json; charset=utf-8",
    )

    gs_path = f"gs://{BUCKET_NAME}/{object_path}"
    return gs_path, object_path


In [17]:
from google.cloud import firestore, storage
import os

def store_statements(
    ticker: str,
    year: int,
    income_statement: dict = None,
    cash_flow: dict = None
):
    filing_doc_id = f"{year}_10K"

    filing_ref = (
        db.collection("companies")
          .document(ticker)
          .collection("filings")
          .document(filing_doc_id)
    )

    # Ensure the filing doc exists
    filing_ref.set({
        "ticker": ticker,
        "year": year,
        "type": "10-K",
        "updatedAt": firestore.SERVER_TIMESTAMP,
    }, merge=True)

    statements_col = filing_ref.collection("statements")

    # Income statement
    if income_statement is not None:
        gs_path, object_path = upload_json_to_firebase_storage(
            ticker=ticker,
            year=year,
            statement_type="incomeStatement",
            data=income_statement,
        )

        statements_col.document("incomeStatement").set({
            "statementType": "incomeStatement",
            "storageGsPath": gs_path,
            "storageObject": object_path,
            "updatedAt": firestore.SERVER_TIMESTAMP,
        }, merge=True)

    # Cash flow
    if cash_flow is not None:
        gs_path, object_path = upload_json_to_firebase_storage(
            ticker=ticker,
            year=year,
            statement_type="cashFlow",
            data=cash_flow,
        )

        statements_col.document("cashFlow").set({
            "statementType": "cashFlow",
            "storageGsPath": gs_path,
            "storageObject": object_path,
            "updatedAt": firestore.SERVER_TIMESTAMP,
        }, merge=True)

    print(f"✅ Stored statements for {ticker} {year}")


In [61]:
# This has been validated to work on 21st Jan 2026

# Pull financial statements for ticker:
financials = get_financials_json("AIG")

# Store, for that ticker, into firestore and firebase storage, the statements pulled from the above 'financials'
store_statements(
    ticker=financials["ticker"],
    year=int(financials["report_date"][:4]),
    income_statement=financials["income_statement"],
    cash_flow=financials["cashflow"]
)


Found 10-K filing: https://www.sec.gov/Archives/edgar/data/5272/000000527226000023/0000005272-26-000023-index.html (filed 2026-02-12, for 2025-12-31)
Using XBRL XML file: https://www.sec.gov/Archives/edgar/data/5272/000000527226000023/aig-20251231_htm.xml
✅ Stored statements for AIG 2025


### [B] Get full 10K as Markdown Text

In [ ]:
# Find primary 10-K HTML and convert it to clean text
import re
import html2text


def get_primary_html_url(index_url):
    """
    Find the main 10-K HTML document inside the filing's index page.
    Returns the actual HTML file URL (not the inline XBRL viewer).
    """
    res = requests.get(index_url, headers=HEADERS)
    if res.status_code != 200:
        raise RuntimeError(f"Failed to fetch index page: {res.status_code}")
    soup = BeautifulSoup(res.text, "html.parser")

    candidates = []

    for link in soup.find_all("a", href=True):
        href = link["href"].strip()
        text = link.get_text(strip=True).lower()
        href_lower = href.lower()

        # Skip XMLs and true exhibits
        if href_lower.endswith(".xml") or any(
            x in href_lower for x in ["_cal.xml", "_lab.xml", "_pre.xml", "_def.xml"]
        ):
            continue
        if "exhibit" in href_lower or text.startswith("ex-") or text.startswith("exhibit"):
            continue

        # Only consider HTML or Inline XBRL viewer links
        if not (
            href_lower.endswith((".htm", ".html"))
            or "ix?doc=" in href_lower
        ):
            continue

        score = 0
        if "10-k" in href_lower or "10k" in href_lower:
            score += 10
        if "10-k" in text or "10k" in text or "form 10-k" in text:
            score += 8
        if "ix?doc=" in href_lower:
            score += 12  # primary document is commonly linked via iXBRL viewer
        # Deprioritize XBRL "R" exhibit renderings (R1.htm, R2.htm, ...)
        if re.search(r"/r\d+\.htm", href_lower):
            score -= 20
        if "index" in href_lower:
            score -= 5

        candidates.append((score, href))

    if not candidates:
        return None

    best_href = sorted(candidates, key=lambda x: x[0], reverse=True)[0][1]

    # Normalize — unwrap inline XBRL viewer to the raw HTML document
    if "ix?doc=" in best_href:
        best_href = best_href.split("ix?doc=")[-1]

    if best_href.startswith("http"):
        return best_href
    if best_href.startswith("/"):
        return "https://www.sec.gov" + best_href
    return "https://www.sec.gov/" + best_href


def clean_10k_html(html_content):
    """Convert messy 10-K HTML into clean text."""
    soup = BeautifulSoup(html_content, "html.parser")

    for tag in soup(["script", "style", "ix:header", "ix:hidden", "link", "meta"]):
        tag.decompose()

    text_maker = html2text.HTML2Text()
    text_maker.ignore_links = True
    text_maker.ignore_images = True
    text_maker.body_width = 0
    clean_text = text_maker.handle(str(soup))

    clean_text = re.sub(r"\n\s*\n", "\n\n", clean_text)
    return clean_text.strip()


### [B1] Store 10K into firebase storage and the pointer to this in firestore

In [ ]:
# Store cleaned 10-K text in Firebase Storage + Firestore pointers
import hashlib
import os
import requests
from dotenv import load_dotenv
from google.cloud import firestore, storage
from google.oauth2 import service_account

load_dotenv()


def _get_firebase_clients():
    """Reuse notebook-global clients when available; otherwise init from .env."""
    global db, storage_client, BUCKET_NAME

    bucket_name = globals().get("BUCKET_NAME") or os.getenv("GCP_STORAGE_BUCKET")
    existing_db = globals().get("db")
    existing_storage = globals().get("storage_client")

    if existing_db is not None and existing_storage is not None and bucket_name:
        return existing_db, existing_storage, bucket_name

    service_account_path = os.getenv("FIREBASE_SERVICE_ACCOUNT_JSON")
    project_id = os.getenv("GCP_PROJECT_ID")
    if not service_account_path:
        raise ValueError("Missing FIREBASE_SERVICE_ACCOUNT_JSON in .env")
    if not project_id:
        raise ValueError("Missing GCP_PROJECT_ID in .env")
    if not bucket_name:
        raise ValueError("Missing GCP_STORAGE_BUCKET in .env")

    creds = service_account.Credentials.from_service_account_file(service_account_path)
    db = firestore.Client(credentials=creds, project=project_id)
    storage_client = storage.Client(credentials=creds, project=project_id)
    BUCKET_NAME = bucket_name
    return db, storage_client, bucket_name


def store_10K_text_from_url(ticker, html_url, report_date):
    """
    Fetch a 10-K HTML from a URL, convert it to cleaned text, and upload
    the text to Firebase Storage under:
      filings/{ticker}/{year}/10K.txt

    Also writes/updates:
      companies/{ticker}/filings/{year}_10K
      ingestion/10k/files/{TICKER}_{year}  (so Data Availability "Last updated" refreshes)
    """
    ticker = (ticker or "").strip().upper()
    if not ticker:
        raise ValueError("ticker is required")
    if not report_date:
        raise ValueError("report_date is required")

    # 1) Fetch HTML
    res = requests.get(html_url, headers=HEADERS)
    if res.status_code != 200:
        raise RuntimeError(f"Failed to fetch HTML from SEC: {res.status_code}")

    # 2) Clean to text
    try:
        text_data = clean_10k_html(res.text)
    except Exception as e:
        raise RuntimeError(f"Failed to clean HTML for {ticker}: {e}") from e

    file_hash = hashlib.sha256(text_data.encode("utf-8")).hexdigest()
    year = int(str(report_date)[:4])
    object_path = f"filings/{ticker}/{year}/10K.txt"

    # 3) Firebase clients
    db_client, storage_client_local, bucket_name = _get_firebase_clients()
    bucket = storage_client_local.bucket(bucket_name)
    blob = bucket.blob(object_path)

    # 4) Upload cleaned text
    blob.upload_from_string(text_data, content_type="text/plain; charset=utf-8")
    gs_path = f"gs://{bucket_name}/{object_path}"

    # 5) Filing pointer used by financial / storage lookups
    filing_doc_id = f"{year}_10K"
    db_client.collection("companies").document(ticker).collection("filings").document(
        filing_doc_id
    ).set(
        {
            "ticker": ticker,
            "year": year,
            "type": "10-K",
            "storageGsPath": gs_path,
            "storageBucket": bucket_name,
            "storageObject": object_path,
            "updatedAt": firestore.SERVER_TIMESTAMP,
        },
        merge=True,
    )

    # 6) Ingestion manifest used by Company Data Availability "Last updated"
    #    Keep status=success if already successful so the row stays visible;
    #    update sha256/source so a later Pinecone run will re-ingest if text changed.
    manifest_ref = (
        db_client.collection("ingestion")
        .document("10k")
        .collection("files")
        .document(f"{ticker}_{year}")
    )
    existing = manifest_ref.get()
    existing_data = existing.to_dict() if existing.exists else {}
    status = existing_data.get("status") or "success"

    manifest_ref.set(
        {
            "ticker": ticker,
            "year": year,
            "sourceGsPath": gs_path,
            "sha256": file_hash,
            "status": status,
            "updatedAt": firestore.SERVER_TIMESTAMP,
        },
        merge=True,
    )

    print(f"✅ Uploaded 10-K to Storage: {gs_path}")
    print(f"✅ Linked in Firestore: /companies/{ticker}/filings/{filing_doc_id}")
    print(f"✅ Updated ingestion manifest: /ingestion/10k/files/{ticker}_{year}")
    if existing_data.get("sha256") and existing_data.get("sha256") != file_hash:
        print(
            "ℹ️ Text hash changed — re-run Pinecone ingestion if you want RAG "
            "chunks to match this new 10-K text."
        )

    return gs_path


#### Try running the 10K storage end-to-end

In [ ]:
# End-to-end: download latest 10-K text for a ticker and store it
# Optional: pass year=2025 to get_latest_10k_url(cik, year=2025)
ticker = "A"

cik = get_cik(ticker)
if not cik:
    raise ValueError(f"CIK not found for ticker {ticker}")

filing_url, filing_date, report_date = get_latest_10k_url(cik)
if not filing_url:
    raise ValueError(f"No 10-K filing found for {ticker}")

print(f"Filing index: {filing_url}")
print(f"Filed: {filing_date} | Report date: {report_date}")

html_url = get_primary_html_url(filing_url)
if not html_url:
    raise ValueError("Could not find primary 10-K HTML in filing index")

print(f"Primary HTML: {html_url}")
gs_path = store_10K_text_from_url(ticker, html_url, report_date)
print(gs_path)
